# 06 - CI Gate For Data And AI Changes

This notebook models Metatate as a release gate. A proposed SQL model, export job, prompt, or AI workflow change is checked before it ships.

The example is notebook-first, but the same pattern can run in GitHub Actions, dbt, Airflow, or an internal deployment pipeline.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

client = get_client()
mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
print(f"Metatate examples mode: {mode}")


def decision_label(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("decision") or decision.get("overall_decision") or data.get("overall_decision", "UNKNOWN")
    return decision or data.get("overall_decision", "UNKNOWN")


def rationale_text(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("rationale") or data.get("summary") or ""
    return data.get("rationale") or data.get("summary") or ""


def agent_action_text(response):
    action = response.get("data", {}).get("agent_action", {})
    if isinstance(action, dict):
        return action.get("message") or action.get("safe_next_step") or action.get("suggested_next_tool") or ""
    return ""


## Proposed changes


In [ ]:
proposed_changes = [
    {
        "change_id": "chg-001",
        "kind": "sql_model",
        "description": "Revenue dashboard aggregates active ARR by region.",
        "sql": "SELECT region, account_status, SUM(arr) AS arr FROM ACMECLOUD_DEMO.PUBLIC.CUSTOMERS WHERE account_status = 'active' GROUP BY region, account_status",
        "operation": "read",
        "intended_use": "analytics",
        "actor_role": "DATA_ANALYST",
    },
    {
        "change_id": "chg-002",
        "kind": "sql_model",
        "description": "Marketing activation model selects names and emails.",
        "sql": "SELECT customer_name, email FROM ACMECLOUD_DEMO.PUBLIC.CUSTOMERS WHERE account_status = 'active'",
        "operation": "read",
        "intended_use": "marketing",
        "actor_role": "GROWTH_ANALYST",
    },
    {
        "change_id": "chg-003",
        "kind": "export_job",
        "description": "CRM sync sends approved customer fields to Salesforce.",
        "table_name": "ACMECLOUD_DEMO.PUBLIC.CUSTOMERS",
        "operation": "export",
        "intended_use": "external_sharing",
        "actor_role": "DATA_ENGINEER",
        "columns": ["CUSTOMER_ID", "CUSTOMER_NAME", "EMAIL", "ACCOUNT_STATUS"],
        "destination": {"system": "SALESFORCE", "jurisdiction": "US"},
        "consumer_jurisdiction": "EU",
    },
]


## Evaluate the gate


In [ ]:
def evaluate_change(change):
    if change["kind"] == "sql_model":
        response = client.validate_query_context(
            change["sql"],
            operation=change["operation"],
            intended_use=change["intended_use"],
            actor_role=change.get("actor_role"),
        )
        decision = response["data"].get("decision", {}).get("decision")
        rationale = response["data"].get("decision", {}).get("rationale")
        action = response["data"].get("agent_action", {}).get("message")
    else:
        response = client.authorize_use(
            change["table_name"],
            operation=change["operation"],
            intended_use=change["intended_use"],
            actor_role=change.get("actor_role"),
            columns=change.get("columns"),
            destination=change.get("destination"),
            consumer_jurisdiction=change.get("consumer_jurisdiction"),
        )
        decision = response["data"].get("decision")
        rationale = response["data"].get("rationale")
        action = response["data"].get("agent_action", {}).get("message")

    if decision == "DENY":
        gate = "fail"
    elif decision == "CONDITIONAL":
        gate = "needs_controls"
    else:
        gate = "pass"

    return {
        "change_id": change["change_id"],
        "kind": change["kind"],
        "decision": decision,
        "gate": gate,
        "rationale": rationale,
        "action": action,
    }

gate_results = pd.DataFrame([evaluate_change(change) for change in proposed_changes])
gate_results[["change_id", "kind", "decision", "gate", "rationale"]]


In [ ]:
blocking = gate_results[gate_results["gate"] == "fail"]
controls = gate_results[gate_results["gate"] == "needs_controls"]

print("Release gate summary")
print(f"pass: {(gate_results['gate'] == 'pass').sum()}")
print(f"needs_controls: {len(controls)}")
print(f"fail: {len(blocking)}")

if len(blocking):
    print("\nBlocking changes")
    print(blocking[["change_id", "decision", "action"]].to_string(index=False))
    if os.getenv("METATATE_EXAMPLES_STRICT_CI_GATE") == "1":
        raise AssertionError("Release gate failed. Resolve denied changes before deployment.")
    print("\nStrict mode is off. Set METATATE_EXAMPLES_STRICT_CI_GATE=1 to make this cell fail like CI.")


## CI-friendly non-throwing summary


In [ ]:
# CI systems often want a machine-readable payload even when a gate fails.
# This cell does not raise; it prints the same result as JSON for logging.
print(json.dumps(gate_results.to_dict(orient="records"), indent=2))


In a real pipeline, the failing change would stop the deployment. Conditional changes can create approval tasks or require anonymization before the release continues.
